# 01 - Explore the property data

This notebook checks the listings and user interactions before any model work. The main checks cover data quality, price ranges, popular areas and available user history.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from app.data import load_interactions, load_properties
from app.data.clean import clean_inside_airbnb_listings
from app.data.validate import validate_properties

pd.set_option('display.max_columns', 40)
plt.style.use('seaborn-v0_8-whitegrid')

## Load the data

The loader uses processed Inside Airbnb data when it exists. Otherwise, it uses the small sample files from the repository. The next step checks the schema and reviews what the cleaning function changes.

In [ ]:
real_properties = project_root / 'data/processed/properties.csv'
real_interactions = project_root / 'data/processed/interactions.csv'
using_real_data = real_properties.exists() and real_interactions.exists()

properties = load_properties(real_properties if using_real_data else None)
interactions = load_interactions(real_interactions if using_real_data else None)

source_name = 'processed Inside Airbnb data' if using_real_data else 'repository sample data'
print(f'Data source: {source_name}')
print(f'Listings: {len(properties):,}')
print(f'Interactions: {len(interactions):,}')

validate_properties(properties)
print('Property schema check: passed')

raw_listings_path = project_root / 'data/raw/listings.csv.gz'
if raw_listings_path.exists():
    raw_sample = pd.read_csv(raw_listings_path, nrows=10_000)
    cleaned_sample = clean_inside_airbnb_listings(raw_sample)
    cleaning_audit = pd.Series({
        'raw_rows_checked': len(raw_sample),
        'clean_rows_kept': len(cleaned_sample),
        'duplicate_ids_removed': raw_sample['id'].duplicated().sum(),
        'rows_with_missing_price': raw_sample['price'].isna().sum(),
        'rows_outside_price_rule': (~cleaned_sample['price'].between(20, 1500)).sum(),
    })
    display(cleaning_audit.to_frame('rows'))
else:
    print('Raw listings are not present. The loaded sample is already cleaned.')

## First look

In [ ]:
display(properties.head(3))
display(interactions.head(3))

In [ ]:
column_types = properties.dtypes.astype(str).rename('dtype').to_frame()
column_types['unique_values'] = properties.apply(lambda column: column.astype(str).nunique(dropna=False))
display(column_types)

## Data quality

These checks catch common problems before they affect retrieval or ranking.

In [ ]:
missing = properties.isna().sum()
quality_report = pd.DataFrame({
    'missing': missing,
    'missing_pct': (missing / len(properties) * 100).round(2),
    'unique': properties.apply(lambda column: column.astype(str).nunique(dropna=False)),
}).sort_values(['missing_pct', 'unique'], ascending=[False, False])
display(quality_report.head(15))

In [ ]:
checks = pd.Series({
    'duplicate_property_ids': properties['property_id'].duplicated().sum(),
    'duplicate_interactions': interactions.duplicated(['user_id', 'property_id', 'timestamp']).sum(),
    'non_positive_prices': (properties['price'] <= 0).sum(),
    'ratings_outside_0_to_5': (~properties['rating'].between(0, 5)).sum(),
    'invalid_latitude': (~properties['latitude'].between(-90, 90)).sum(),
    'invalid_longitude': (~properties['longitude'].between(-180, 180)).sum(),
    'interactions_without_listing': (~interactions['property_id'].isin(properties['property_id'])).sum(),
}, name='rows')
display(checks.to_frame())

## Price and listing quality

In [ ]:
summary_columns = ['price', 'bedrooms', 'rating', 'review_count', 'availability_365']
display(properties[summary_columns].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
price_limit = properties['price'].quantile(0.95)
properties.loc[properties['price'] <= price_limit, 'price'].plot(
    kind='hist', bins=30, ax=axes[0], color='#047857', title='Nightly price (up to 95th percentile)'
)
properties.loc[properties['rating'] > 0, 'rating'].plot(
    kind='hist', bins=20, ax=axes[1], color='#2563eb', title='Guest rating'
)
axes[0].set_xlabel('Price in EUR')
axes[1].set_xlabel('Rating')
plt.tight_layout()
plt.show()

In [ ]:
area_summary = (
    properties.groupby('neighborhood')
    .agg(listings=('property_id', 'count'), median_price=('price', 'median'), median_rating=('rating', 'median'))
    .sort_values('listings', ascending=False)
    .head(12)
)
display(area_summary.round(2))
display(properties['property_type'].value_counts().rename('listings').to_frame())

## Interaction behaviour

Reviews are used as positive feedback. They are useful, but they are not the same as clicks, saves or bookings.

In [ ]:
user_activity = interactions.groupby('user_id').size().rename('interactions')
item_activity = interactions.groupby('property_id').size().rename('interactions')
possible_pairs = interactions['user_id'].nunique() * properties['property_id'].nunique()
sparsity = 1 - interactions[['user_id', 'property_id']].drop_duplicates().shape[0] / max(possible_pairs, 1)

interaction_summary = pd.Series({
    'users': interactions['user_id'].nunique(),
    'listings_with_feedback': interactions['property_id'].nunique(),
    'median_interactions_per_user': user_activity.median(),
    'users_with_2_or_more_events': (user_activity >= 2).sum(),
    'catalog_sparsity_pct': round(sparsity * 100, 2),
    'first_event': interactions['timestamp'].min(),
    'last_event': interactions['timestamp'].max(),
})
display(interaction_summary.to_frame('value'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
user_activity.clip(upper=user_activity.quantile(0.99)).plot(
    kind='hist', bins=30, ax=axes[0], color='#7c3aed', title='Interactions per user'
)
item_activity.sort_values(ascending=False).head(15).plot(
    kind='bar', ax=axes[1], color='#d97706', title='Most reviewed listings'
)
axes[0].set_xlabel('Interaction count')
axes[1].set_xlabel('Property ID')
plt.tight_layout()
plt.show()

## Notes after exploration

- Use hard limits for city, availability and unrealistic prices.
- Keep price as a ranking feature because budget is an important user need.
- Use metadata for new users because many users have little history.
- Use a time split for evaluation so future feedback does not enter training.
- Treat review history as a public feedback proxy, not as full booking behaviour.